<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week8/Week8_Feature_Importance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
# regression example

# put model in eval mode
model.eval()

# get baseline predictions for baseline MSE
with torch.no_grad():
    baseline_preds = model(X_val).squeeze()

# baseline MSE calculation
baseline_mse = ((baseline_preds - y_val) ** 2).mean().item()

# number of times to repeat permutations
n_repeats = 10

# to store final averaged importances
feature_importances = []

# loop through each feature
for j in range(X_val.shape[1]):

    repeat_scores = []

    # repeat permutations multiple times (less noise)
    for _ in range(n_repeats):

        # copy validation data for permutation
        X_perm = X_val.clone()

        # randomly permute j-th column
        idx = torch.randperm(X_perm.shape[0])
        X_perm[:, j] = X_perm[idx, j]

        # get predictions on permuted data
        with torch.no_grad():
            perm_preds = model(X_perm).squeeze()

        # compute permuted MSE
        perm_mse = ((perm_preds - y_val) ** 2).mean().item()

        # importance = increase in MSE (new minus baseline)
        importance = perm_mse - baseline_mse

        repeat_scores.append(importance)

    # average across repeats
    avg_importance = sum(repeat_scores) / len(repeat_scores)

    feature_importances.append(avg_importance)

feature_names = ['danceability', 'energy', 'loudness', 'tempo', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']

# sort features by importance
pairs = list(zip(feature_names, feature_importances))
pairs.sort(key = lambda x: x[1], reverse = True) # sort by largest to smallest

print('Regression Permutation Importances:\n')

for name, imp in pairs:
    print(f'{name:<20} {imp:.6f}')

'''


'''
# classification example

# put model in eval mode
model.eval()

# baseline predictions to compute accuracy
with torch.no_grad():
    baseline_logits = model(X_val)
    baseline_preds = baseline_logits.argmax(dim = 1) # largest logit = predicted class

# compute accuracy baseline
baseline_acc = (baseline_preds == y_val).float().mean().item()

# number of repeated permutations
n_repeats = 10

# to store final averaged importances
feature_importances = []

# loop through each feature
for j in range(X_val.shape[1]):

    repeat_scores = []

    # repeat permutations multiple times (less noise)
    for _ in range(n_repeats):

        # copy validation data for permutation
        X_perm = X_val.clone()

        # permute j-th feature column
        idx = torch.randperm(X_perm.shape[0])
        X_perm[:, j] = X_perm[idx, j]

        # predictions on permuted data
        with torch.no_grad():
            perm_logits = model(X_perm)
            perm_preds = perm_logits.argmax(dim=1)

        # compute accuracy
        perm_acc = (perm_preds == y_val).float().mean().item()

        # importance = drop in accuracy (baseline minus new)
        importance = baseline_acc - perm_acc

        repeat_scores.append(importance)

    # average across repeats
    avg_importance = sum(repeat_scores) / len(repeat_scores)

    feature_importances.append(avg_importance)

# feature names
feature_names = ['danceability', 'energy', 'loudness', 'tempo', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']

# sort by importance
pairs = list(zip(feature_names, feature_importances))
pairs.sort(key = lambda x: x[1], reverse = True) # sort by largest to smallest

print('Classification Permutation Importances:\n')

for name, imp in pairs:
    print(f'{name:<20} {imp:.6f}')

'''

'''
# check coefficients for sklearn linear/ridge regression model

# feature names
feature_names = ['danceability', 'energy', 'loudness', 'tempo', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']
for name, coef in zip(feature_names, model.coef_):
    print(f'{name:<20} {coef:.6f}')
'''

'''
# check coefficients for class K in sklearn softmax regression model

print(model.coef_[k])

'''

'''
# xgboost gain importance


booster = model.get_booster()
importance = booster.get_score(importance_type = 'gain')

feature_names = ['danceability', 'energy', 'loudness', 'tempo', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']

pairs = []

for i, name in enumerate(feature_names):

    key = f'f{i}' # labeled f0,...,fp

    gain = importance.get(key, 0) # return 0 if missing

    pairs.append((name, gain))

pairs.sort(key = lambda x: x[1], reverse = True)
for name, gain in pairs:
    print(f'{name:<20} {gain:.4f}')

'''